# Building a Transformer From Scratch

Decoder-only (GPT-style) transformer, following `learning_material/transformer-roadmap.md`.

In [ ]:
## loading the dataset
with open('../data/dataset.txt', 'r', encoding='utf-8') as f:
    dataset = f.read()

print(dataset[:30])

In [ ]:
## vocabulary: every unique character -> an integer
uchars = sorted(list(set(dataset)))
vocab_size = len(uchars)
print(vocab_size, 'unique characters')
print(''.join(uchars))

stoi = {ch: i for i, ch in enumerate(uchars)}  # char -> int
itos = {i: ch for i, ch in enumerate(uchars)}  # int -> char

encode = lambda s: [stoi[c] for c in s]
decode = lambda n: ''.join([itos[i] for i in n])

print(encode('Hi, if you are reading this, hire me'))
print(decode(encode('Hi, if you are reading this, hire me')))

In [ ]:
import torch

data = torch.tensor(encode(dataset), dtype=torch.long)

## train / validation split
n = int(0.95 * len(data))   # 95% train, 5% validation
train_data = data[:n]
val_data = data[n:]
print(len(train_data), 'train tokens |', len(val_data), 'val tokens')

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32      # sequences processed in parallel
block_size = 64      # maximum context length
max_iters = 3000
eval_interval = 300
eval_iters = 100
learning_rate = 1e-3
n_embd = 128         # embedding dimension
n_head = 4           # attention heads
n_layer = 4          # transformer blocks
dropout = 0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(1337)

In [ ]:
## grab a batch of (input, next-char target) pairs
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))  # random start positions
    x = torch.stack([d[i:i + block_size] for i in ix])          # inputs
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # targets = input shifted by 1
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('inputs ', tuple(xb.shape))
print('targets', tuple(yb.shape))

## Attention

For each token we compute a query (what am I looking for), a key (what do I offer) and a value (what I contribute). Score = query dot key, scaled by 1/sqrt(d_k) for stability, softmax to weights, weighted sum of values. The causal mask blocks future tokens so the model can't cheat when predicting the next character.

In [ ]:
class Head(nn.Module):
    """one head of self-attention"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # lower-triangular mask: token t may only attend to tokens <= t
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # scaled dot-product attention
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # causal mask
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)  # (B, T, head_size)
        return wei @ v     # (B, T, head_size)

In [ ]:
class MultiHeadAttention(nn.Module):
    """several attention heads in parallel, concatenated and projected"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

## The model

A block does attention (tokens talk to each other) then a feed-forward MLP (each position thinks on its own), each wrapped in a residual connection with pre-layer-norm. Stack the blocks, add the final norm and the output head.

In [ ]:
class FeedForward(nn.Module):
    """position-wise MLP: expand 4x, ReLU, project back"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """communicate (attention) then compute (MLP), each with residual + pre-LN"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # residual around attention
        x = x + self.ffwd(self.ln2(x))  # residual around MLP
        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)                                # (B, T, C)
        pos = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        x = self.blocks(tok + pos)   # add position info, then run the blocks
        x = self.ln_f(x)
        logits = self.lm_head(x)     # (B, T, vocab_size)

        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss